# Deepfake Detector v7 - AlexNet

Versao usando **AlexNet** com dataset local balanceado preparado fora do notebook.

Objetivos desta versao:
- Treinar com um conjunto mais proximo do fluxo real do projeto
- Reforcar grupos sub-representados no conjunto real
- Manter o notebook v6 intacto para comparacao historica

Dataset usado nesta versao:
- **Origem**: `data/dataset_colab` preparado localmente no projeto
- **Formato**: `train/real`, `train/fake`, `val/real`, `val/fake`, `test/real`, `test/fake`
- **Balanceamento**: 6000 reais e 6000 fake



In [ ]:
# Instalar dependencias
!pip install -q torch torchvision scikit-learn pillow tqdm



In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import classification_report, roc_auc_score, f1_score, precision_recall_curve
from tqdm.auto import tqdm

# Seed para reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {device}')



## 1. Download dos Datasets

In [ ]:
# Opcional: montar Google Drive no Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive montado com sucesso.')
except Exception as e:
    print(f'Google Drive nao montado: {e}')



In [ ]:
# Configurar caminho do dataset preparado localmente
DATASET_ROOT = '/content/drive/MyDrive/data/dataset_colab'

if not os.path.exists(DATASET_ROOT):
    raise FileNotFoundError(
        f'Dataset nao encontrado em {DATASET_ROOT}. '
        'Suba a pasta data/dataset_colab para o Google Drive antes de treinar.'
    )

print(f'Dataset encontrado em: {DATASET_ROOT}')



In [ ]:
def collect_images(root_dir, extensions=('.jpg', '.jpeg', '.png', '.webp', '.bmp')):
    """Coleta todos os caminhos de imagem recursivamente."""
    images = []
    for root, dirs, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith(extensions):
                images.append(os.path.join(root, f))
    return images

train_real = collect_images(os.path.join(DATASET_ROOT, 'train', 'real'))
train_fake = collect_images(os.path.join(DATASET_ROOT, 'train', 'fake'))
val_real = collect_images(os.path.join(DATASET_ROOT, 'val', 'real'))
val_fake = collect_images(os.path.join(DATASET_ROOT, 'val', 'fake'))
test_real = collect_images(os.path.join(DATASET_ROOT, 'test', 'real'))
test_fake = collect_images(os.path.join(DATASET_ROOT, 'test', 'fake'))

print(f'Train real: {len(train_real)}')
print(f'Train fake: {len(train_fake)}')
print(f'Val real: {len(val_real)}')
print(f'Val fake: {len(val_fake)}')
print(f'Test real: {len(test_real)}')
print(f'Test fake: {len(test_fake)}')



In [ ]:
# Construir splits a partir do dataset ja preparado
train_data = [(p, 0) for p in train_real] + [(p, 1) for p in train_fake]
val_data = [(p, 0) for p in val_real] + [(p, 1) for p in val_fake]
test_data = [(p, 0) for p in test_real] + [(p, 1) for p in test_fake]

random.shuffle(train_data)
random.shuffle(val_data)
random.shuffle(test_data)

print(f'Train: {len(train_data)} ({sum(x[1] for x in train_data)} fakes)')
print(f'Val: {len(val_data)} ({sum(x[1] for x in val_data)} fakes)')
print(f'Test: {len(test_data)} ({sum(x[1] for x in test_data)} fakes)')



In [ ]:
# Resumo do dataset v7
all_images = train_data + val_data + test_data
all_reals = [path for path, label in all_images if label == 0]
all_fakes = [path for path, label in all_images if label == 1]

print(f'Total imagens: {len(all_images)}')
print(f'Total reais: {len(all_reals)}')
print(f'Total fakes: {len(all_fakes)}')
print(f'Ratio fake/real: {len(all_fakes)/len(all_reals):.2f}')



## 2. Dataset e Transforms

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            # Em caso de imagem corrompida, retorna outra
            new_idx = (idx + 1) % len(self.data)
            return self.__getitem__(new_idx)

# Transforms - AlexNet espera 224x224 (mesmo que EfficientNet)
# Augmentation agressivo para melhor generalização
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Criar datasets
train_dataset = DeepfakeDataset(train_data, train_transform)
val_dataset = DeepfakeDataset(val_data, val_transform)
test_dataset = DeepfakeDataset(test_data, val_transform)

print(f"Datasets criados: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")

In [ ]:
# DataLoaders
BATCH_SIZE = 64  # AlexNet é mais leve, pode usar batch maior

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

## 3. Modelo AlexNet

In [ ]:
class DeepfakeDetectorAlexNet(nn.Module):
    """
    Detector de Deepfake usando AlexNet como backbone.
    
    AlexNet Features:
    - Conv layers: 5 camadas convolucionais
    - Classifier: 3 camadas fully connected (originalmente para 1000 classes)
    - Output features: 4096 (da penúltima FC layer)
    """
    
    def __init__(self, pretrained=True, dropout=0.5):
        super().__init__()
        
        # Carregar AlexNet pré-treinado no ImageNet
        if pretrained:
            weights = models.AlexNet_Weights.IMAGENET1K_V1
            self.backbone = models.alexnet(weights=weights)
        else:
            self.backbone = models.alexnet(weights=None)
        
        # AlexNet original tem:
        # features (conv layers) -> avgpool -> classifier (FC layers)
        # classifier[6] é a última FC (4096 -> 1000)
        
        # Extrair features (conv + avgpool)
        self.features = self.backbone.features
        self.avgpool = self.backbone.avgpool
        
        # Número de features após flatten: 256 * 6 * 6 = 9216
        num_features = 256 * 6 * 6
        
        # Novo classifier para detecção de deepfake (binário)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(num_features, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(4096, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(1024, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, 1)
        )
        
        # Inicializar novas camadas
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Inicializar pesos das novas camadas."""
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # Feature extraction
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        
        # Classification
        x = self.classifier(x)
        return x.squeeze(-1)
    
    def predict_proba(self, x):
        """Retorna probabilidade de ser fake."""
        with torch.no_grad():
            logits = self.forward(x)
            probs = torch.sigmoid(logits)
        return probs


model = DeepfakeDetectorAlexNet(pretrained=True, dropout=0.5).to(device)

# Contar parâmetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Modelo AlexNet criado:")
print(f"  Total de parâmetros: {total_params:,}")
print(f"  Parâmetros treináveis: {trainable_params:,}")

## 4. Treinamento

In [ ]:
# Focal Loss para lidar com exemplos difíceis
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(inputs, targets.float(), reduction='none')
        pt = torch.exp(-bce)
        focal_weight = (1 - pt) ** self.gamma
        
        # Alpha balancing
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        
        return (alpha_weight * focal_weight * bce).mean()

criterion = FocalLoss(alpha=0.5, gamma=2.0)

# Optimizer com learning rate um pouco maior para AlexNet
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

# Scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).long()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        pbar.set_postfix({'loss': loss.item(), 'acc': correct/total})
    
    return total_loss / len(loader), correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    total_loss = 0
    
    for images, labels in tqdm(loader, desc='Evaluating'):
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).long()
        
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    acc = (all_preds == all_labels).mean()
    auc = roc_auc_score(all_labels, all_probs)
    f1 = f1_score(all_labels, all_preds)
    
    return total_loss / len(loader), acc, auc, f1, all_probs, all_labels

In [ ]:
# Treinamento
EPOCHS = 20  # Mais epochs para AlexNet convergir bem
best_auc = 0
best_model_state = None
patience = 7
no_improve = 0

print("="*60)
print("TREINAMENTO - AlexNet Deepfake Detector v6")
print("="*60)

for epoch in range(EPOCHS):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{EPOCHS} | LR: {optimizer.param_groups[0]['lr']:.2e}")
    print(f"{'='*50}")
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_auc, val_f1, _, _ = evaluate(model, val_loader)
    
    scheduler.step()
    
    print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
    print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, AUC: {val_auc:.4f}, F1: {val_f1:.4f}")
    
    if val_auc > best_auc:
        best_auc = val_auc
        best_model_state = model.state_dict().copy()
        print(f"✓ Novo melhor modelo! AUC: {best_auc:.4f}")
        no_improve = 0
    else:
        no_improve += 1
        print(f"Sem melhora ({no_improve}/{patience})")
    
    if no_improve >= patience:
        print("\n⚠ Early stopping!")
        break

# Carregar melhor modelo
model.load_state_dict(best_model_state)
print(f"\n✓ Melhor modelo carregado (AUC: {best_auc:.4f})")

## 5. Calibração de Threshold

In [ ]:
# Avaliar no conjunto de validação para calibrar threshold
_, _, _, _, val_probs, val_labels = evaluate(model, val_loader)

# Encontrar threshold que maximiza F1
precision, recall, thresholds = precision_recall_curve(val_labels, val_probs)

# Calcular F1 para cada threshold
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

# Threshold conservador (alta precisão para minimizar falsos positivos)
high_precision_idx = np.where(precision >= 0.9)[0]
if len(high_precision_idx) > 0:
    conservative_threshold = thresholds[high_precision_idx[0]] if high_precision_idx[0] < len(thresholds) else 0.7
else:
    conservative_threshold = 0.7

print(f"Threshold ótimo (F1): {best_threshold:.4f}")
print(f"Threshold conservador (precisão ≥90%): {conservative_threshold:.4f}")

# Usar threshold conservador para evitar falsos positivos
final_threshold = conservative_threshold
print(f"\nUsando threshold final: {final_threshold:.4f}")

## 6. Avaliação Final

In [ ]:
# Avaliar no conjunto de teste com threshold calibrado
test_loss, test_acc, test_auc, test_f1, test_probs, test_labels = evaluate(model, test_loader)

# Calcular métricas com threshold conservador
test_preds_calibrated = (test_probs > final_threshold).astype(int)

print("\n" + "="*60)
print("RESULTADOS FINAIS - AlexNet (Conjunto de Teste)")
print("="*60)
print(f"AUC-ROC: {test_auc:.4f}")
print(f"Threshold: {final_threshold:.4f}")
print("\nCom threshold padrão (0.5):")
print(f"  Accuracy: {test_acc:.4f}")
print(f"  F1-Score: {test_f1:.4f}")
print(f"\nCom threshold conservador ({final_threshold:.2f}):")
calibrated_acc = (test_preds_calibrated == test_labels).mean()
calibrated_f1 = f1_score(test_labels, test_preds_calibrated)
print(f"  Accuracy: {calibrated_acc:.4f}")
print(f"  F1-Score: {calibrated_f1:.4f}")

print("\n" + "="*60)
print("RELATÓRIO DETALHADO")
print("="*60)
print(classification_report(test_labels, test_preds_calibrated, target_names=['REAL', 'FAKE']))

In [ ]:
# Analisar taxas de erro por fonte
print("\n" + "="*60)
print("ANÁLISE DE FALSOS POSITIVOS")
print("="*60)

# Separar reais e fakes do teste
real_indices = [i for i, (path, label) in enumerate(test_data) if label == 0]
fake_indices = [i for i, (path, label) in enumerate(test_data) if label == 1]

real_probs = test_probs[real_indices]
fake_probs = test_probs[fake_indices]

print(f"\nImagens REAIS (n={len(real_probs)}):")
print(f"  Média de p_fake: {real_probs.mean():.4f}")
print(f"  Mediana de p_fake: {np.median(real_probs):.4f}")
print(f"  Max p_fake: {real_probs.max():.4f}")
fps = (real_probs > final_threshold).sum()
print(f"  Falsos positivos (>{final_threshold:.2f}): {fps} de {len(real_probs)} ({100*fps/len(real_probs):.1f}%)")

print(f"\nImagens FAKE (n={len(fake_probs)}):")
print(f"  Média de p_fake: {fake_probs.mean():.4f}")
print(f"  Mediana de p_fake: {np.median(fake_probs):.4f}")
print(f"  Min p_fake: {fake_probs.min():.4f}")
fns = (fake_probs < final_threshold).sum()
print(f"  Falsos negativos (<{final_threshold:.2f}): {fns} de {len(fake_probs)} ({100*fns/len(fake_probs):.1f}%)")

## 7. Comparação com EfficientNet

In [ ]:
# Comparação de características
print("\n" + "="*60)
print("COMPARAÇÃO: AlexNet vs EfficientNet-B0")
print("="*60)

comparison_data = {
    'Métrica': ['Parâmetros', 'Ano', 'ImageNet Top-1 Acc', 'Velocidade Inferência'],
    'AlexNet': ['~61M', '2012', '~56%', 'Mais rápido'],
    'EfficientNet-B0': ['~5.3M', '2019', '~77%', 'Moderado']
}

print(f"\n{'Métrica':<25} {'AlexNet':<20} {'EfficientNet-B0':<20}")
print("-" * 65)
for i in range(len(comparison_data['Métrica'])):
    print(f"{comparison_data['Métrica'][i]:<25} {comparison_data['AlexNet'][i]:<20} {comparison_data['EfficientNet-B0'][i]:<20}")

print(f"\n\nResultados deste treinamento (AlexNet):")
print(f"  AUC-ROC: {test_auc:.4f}")
print(f"  F1-Score: {calibrated_f1:.4f}")
print(f"  Accuracy: {calibrated_acc:.4f}")

## 8. Salvar Modelo

In [ ]:
# Salvar checkpoint completo
checkpoint = {
    'model_state_dict': best_model_state,
    'best_auc': float(best_auc),
    'best_threshold': float(final_threshold),
    'test_auc': float(test_auc),
    'test_f1': float(calibrated_f1),
    'test_accuracy': float(calibrated_acc),
    'config': {
        'model_name': 'alexnet',
        'version': 'v7',
        'dropout': 0.5,
        'datasets': ['dataset_colab'],
        'description': 'AlexNet treinado com dataset local balanceado preparado no projeto',
        'input_size': 224,
        'num_features': 9216
    }
}

save_path = 'alexnet_deepfake_v7.pth'
torch.save(checkpoint, save_path)
print(f'\n✓ Modelo salvo em: {save_path}')
print(f'  AUC: {best_auc:.4f}')
print(f'  Threshold: {final_threshold:.4f}')
print(f'  Tamanho: {os.path.getsize(save_path) / 1024 / 1024:.1f} MB')



In [ ]:
# Instrucoes para usar o modelo
print("""
╔══════════════════════════════════════════════════════════════════════╗
║              AlexNet DEEPFAKE DETECTOR v7 - COMPLETO!                ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  Para usar na aplicacao DeepTrustShield:                             ║
║                                                                      ║
║  1. Copie o arquivo para a pasta de pesos:                           ║
║     cp alexnet_deepfake_v7.pth model/weights/                        ║
║                                                                      ║
║  2. Reinicie o worker para carregar o novo modelo                    ║
║                                                                      ║
║  O modelo v7 oferece:                                                ║
║  - Dataset local balanceado e controlado                             ║
║  - Reforco de grupos reais sub-representados                         ║
║  - Split train/val/test preparado fora do notebook                   ║
║  - Separacao clara em relacao ao baseline v6                         ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")

